In [ ]:
# Import libraries and set parameters
import pandas as pd
import rasterio
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import math
from rasterio.enums import Resampling
from rasterio import mask
import geopandas as gpd

# Define configuration parameters
class Config:
    # Path configuration
    excel_path = r"D:\...\03zy_0830_classification_results.xlsx"
    tif_folder = r"D:\...\season_event_count_2015_2024_v4"
    boundary_shp = r"D:\...\ne_110m_coastline.shp"
    mask_shp_path = r"D:\...\WorldMap.shp"
    output_dir = r"C:\Users\Lenovo\Desktop"
    # Calculation parameters
    max_radius = 1000
    total_files = 40
    final_tif_name = "OverblownEvents_251009.tif"

# Utility functions
def latlon_to_grid(transform, x, y):
    """Convert geographic coordinates to raster row and column indices"""
    col, row = ~transform * (x, y)
    return int(row), int(col)

In [ ]:
# Build coordinate dictionary grouped by quarter
df = pd.read_excel(Config.excel_path, engine='openpyxl')
df = df.rename(columns={'lon': 'X', 'lat': 'Y'})
df['radius'] = df['radius'].clip(upper=Config.max_radius)

coord_dict = defaultdict(list)
for _, row in df.iterrows():
    coord_dict[row['category']].append((row['X'], row['Y'], row['radius']))

# Get paths of all TIF files
tif_paths = list(Path(Config.tif_folder).glob("*.tif"))
if len(tif_paths) != Config.total_files:
    print(f"Warning: Detected {len(tif_paths)} TIF files, inconsistent with configured {Config.total_files}")

# Initialize ratio sum array with original raster dimensions
with rasterio.open(tif_paths[0]) as src:
    ratio_sum_total = np.zeros((src.height, src.width), dtype=np.float64) 
    meta = src.meta.copy()
    meta.update({'dtype': 'float64'})

In [ ]:
import math
from tqdm import tqdm
import numpy as np
import rasterio

# Initialize two cumulative arrays
with rasterio.open(tif_paths[0]) as src:
    special_report_sum_total = np.zeros((src.height, src.width), dtype=np.float64)
    event_sum_total = np.zeros((src.height, src.width), dtype=np.float64)
    meta = src.meta.copy()
    meta.update(dtype='float64', count=1)

for tif_path in tqdm(tif_paths, desc="Processing TIF by season"):
    season = tif_path.stem
    with rasterio.open(tif_path) as src:
        data = src.read(1)
        transform = src.transform
        count_grid = np.zeros(data.shape, dtype=np.uint8)

        # Get point data for current season
        season_points = coord_dict.get(season, [])
        if not season_points:
            print(f"Warning: No CSV data found for {season}")
        else:
            for x, y, r_km in season_points:
                try:
                    lat = y
                    r_lat = r_km / 111.32
                    r_lon = r_km / (111.32 * math.cos(math.radians(lat)))

                    row_c, col_c = latlon_to_grid(transform, x, y)
                    row_range = math.ceil(r_lat / abs(transform.e))
                    col_range = math.ceil(r_lon / transform.a)

                    min_row, max_row = max(0, row_c - row_range), min(src.height-1, row_c + row_range)
                    min_col, max_col = max(0, col_c - col_range), min(src.width-1, col_c + col_range)

                    for row in range(min_row, max_row+1):
                        for col in range(min_col, max_col+1):
                            px, py = transform * (col+0.5, row+0.5)
                            dx = (px - x) * 111.32 * math.cos(math.radians(lat))
                            dy = (py - y) * 111.32
                            if math.hypot(dx, dy) <= r_km:
                                count_grid[row, col] += 1
                except Exception as e:
                    print(f"Error processing point ({x},{y}): {str(e)}")
                    continue

        # Binarization
        count_grid = np.where(count_grid > 0, 1, 0)

        # Boolean masks
        data_bool = data.astype(bool)
        count_bool = count_grid.astype(bool)

        # Calculate special report and event counts
        special_report = (~data_bool) & count_bool
        events = data_bool

        # Accumulate results
        special_report_sum_total += special_report
        event_sum_total += events

# Calculate per-pixel ratio
ratio = np.divide(special_report_sum_total, event_sum_total, 
                 out=np.zeros_like(special_report_sum_total, dtype=np.float64), 
                 where=event_sum_total != 0)

In [ ]:
# Apply geographic mask
mask_shp = gpd.read_file(Config.mask_shp_path).to_crs(meta['crs'])

with rasterio.io.MemoryFile() as memfile:
    with memfile.open(**meta) as dst:
        dst.write(ratio, 1)
    with memfile.open() as src:
        masked_result, masked_transform = mask.mask(
            src,
            mask_shp.geometry,
            crop=True,
            nodata=np.nan,
            all_touched=True
        )
        # Update metadata after masking
        masked_meta = meta.copy()
        masked_meta.update({
            'height': masked_result.shape[1],
            'width': masked_result.shape[2],
            'transform': masked_transform
        })

# Save final integrated result
output_path = Path(Config.output_dir) / Config.final_tif_name
with rasterio.open(output_path, 'w', **masked_meta) as dst:
    dst.write(masked_result)
print(f"Final result saved to: {output_path}")